In [1]:
import os, sys, json, time, warnings, logging
from pathlib import Path
PROJECT = Path(r"C:\Users\hp\Documents\teliport_prototype")
os.chdir(PROJECT / "notebooks")
sys.path.insert(0, str(PROJECT / "src"))
%load_ext autoreload
%autoreload 2

warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)
import numpy as np, pandas as pd, mlflow
from cmapss_data import load_all, SUBSETS

DATA_DIR = PROJECT / "data" / "raw" / "CMAPSS"
INCOMING = PROJECT / "data" / "incoming"
MODELS = PROJECT / "models"
REPORTS = PROJECT / "reports"
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
data = load_all(DATA_DIR)
print("Working dir:", Path.cwd())

Working dir: C:\Users\hp\Documents\teliport_prototype\notebooks


In [2]:
%%writefile ../src/mlops.py
"""MLOps: MLflow experiment tracking, model registry, drift check and retraining flow."""
from __future__ import annotations

import json
import tempfile
from datetime import datetime, timezone
from pathlib import Path

import joblib
import mlflow
import numpy as np
import pandas as pd
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

from anomaly import HealthMonitor, detection_report
from cmapss_data import COLUMNS, OP_COLS, SENSOR_COLS
from features import CMAPSSPreprocessor, DEFAULT_RUL_CAP, last_cycle
from modeling import LGB_PARAMS, evaluate, fit_model, make_lgbm, predict

EXPERIMENT = "teliport-predictive-maintenance"


def clean_key(k):
    """MLflow metric names cannot contain '<=' or '%'."""
    return k.replace("<=", "_le_").replace("%", "pct")


def model_name(fd):
    return f"teliport_rul_{fd}"


def setup_mlflow(project):
    project = Path(project)
    mlflow.set_tracking_uri(f"sqlite:///{(project / 'mlflow.db').as_posix()}")
    if mlflow.get_experiment_by_name(EXPERIMENT) is None:
        mlflow.create_experiment(EXPERIMENT, artifact_location=(project / "mlruns").as_uri())
    mlflow.set_experiment(EXPERIMENT)
    return MlflowClient()


# ---------------- data checks ----------------
def validate_batch(df):
    """Schema + sanity checks before new data is allowed into training."""
    issues = []
    missing = [c for c in COLUMNS if c not in df.columns]
    if missing:
        issues.append(f"missing columns: {missing}")
    else:
        if df[SENSOR_COLS + OP_COLS].isna().mean().max() > 0.2:
            issues.append("more than 20% missing values in a column")
        if df.duplicated(["unit", "cycle"]).any():
            issues.append(f"{int(df.duplicated(['unit', 'cycle']).sum())} duplicate (unit, cycle) rows (will be de-duplicated)")
        if (df["cycle"] < 1).any():
            issues.append("cycle numbers < 1")
    blocking = [i for i in issues if "duplicate" not in i]
    return {"ok": not blocking, "issues": issues, "rows": len(df), "engines": int(df["unit"].nunique())}


def psi(ref, new, bins=10):
    """Population Stability Index: <0.1 stable, 0.1-0.25 moderate shift, >0.25 major shift."""
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0
    edges[0], edges[-1] = -np.inf, np.inf
    r = np.histogram(ref, edges)[0] / len(ref) + 1e-6
    n = np.histogram(new, edges)[0] / len(new) + 1e-6
    return float(np.sum((n - r) * np.log(n / r)))


def drift_report(prep, ref_raw, new_raw):
    ref = prep.transform(ref_raw)
    new = prep.transform(new_raw)
    rep = pd.Series({s: psi(ref[f"{s}_z"], new[f"{s}_z"]) for s in prep.sensors}).round(3)
    return rep.sort_values(ascending=False)


# ---------------- train + log + register ----------------
def train_and_log(fd, train_raw, test_raw, test_rul, project, run_name, cap=DEFAULT_RUL_CAP,
                  tags=None, extra_params=None):
    """Fit preprocessor + RUL model + health monitor, evaluate on the golden holdout,
    log everything to MLflow and register a new model version."""
    prep = CMAPSSPreprocessor.for_subset(fd)
    tr = prep.fit_transform(train_raw)
    F = prep.feature_names_
    model = fit_model(tr, F, make_lgbm, cap, weighted=True)
    hm = HealthMonitor().fit(tr, prep.sensors)

    te_last = last_cycle(prep.transform(test_raw)).merge(test_rul[["unit", "RUL"]], on="unit")
    p = predict(model, te_last, F)
    metrics = evaluate(te_last["RUL"], p)
    metrics = {clean_key(k): v for k, v in metrics.items() if k != "n"}
    det = detection_report(tr, hm.transform(tr))

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tags({"subset": fd, **(tags or {})})
        mlflow.log_params({"subset": fd, "rul_cap": cap, "train_engines": int(train_raw["unit"].nunique()),
                           "train_rows": len(train_raw), "n_features": len(F),
                           "anomaly_method": hm.method, "anomaly_quantile": hm.quantile,
                           **{f"lgb_{k}": v for k, v in LGB_PARAMS.items()}, **(extra_params or {})})
        mlflow.log_metrics({f"test_{k}": float(v) for k, v in metrics.items()})
        mlflow.log_metric("anomaly_false_alarm_pct", float(det["false_alarm_rate_%"]))
        with tempfile.TemporaryDirectory() as tmp:
            joblib.dump(prep, Path(tmp) / f"preprocessor_{fd}.joblib")
            joblib.dump(hm, Path(tmp) / f"health_monitor_{fd}.joblib")
            mlflow.log_artifacts(tmp, artifact_path="pipeline")
        info = mlflow.lightgbm.log_model(model, name="rul_model",
                                        signature=infer_signature(tr[F].head(50), model.predict(tr[F].head(50))),
                                        registered_model_name=model_name(fd))
        version = info.registered_model_version
    return {"run_id": run.info.run_id, "version": str(version), "metrics": metrics,
            "prep": prep, "model": model, "hm": hm}


def set_champion(client, fd, version):
    client.set_registered_model_alias(model_name(fd), "champion", str(version))


def get_champion(client, fd):
    try:
        mv = client.get_model_version_by_alias(model_name(fd), "champion")
    except Exception:
        return None
    run = client.get_run(mv.run_id)
    return {"version": mv.version, "run_id": mv.run_id, "metrics": run.data.metrics}


def deploy(result, fd, models_dir):
    """Copy the champion pipeline to the folder that inference / dashboard / agent read from."""
    models_dir = Path(models_dir)
    models_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(result["prep"], models_dir / f"preprocessor_{fd}.joblib")
    joblib.dump(result["model"], models_dir / f"lgbm_{fd}.joblib")
    joblib.dump(result["hm"], models_dir / f"health_monitor_{fd}.joblib")
    (models_dir / f"deployed_{fd}.json").write_text(json.dumps(
        {"subset": fd, "version": result["version"], "run_id": result["run_id"],
         "deployed_at": datetime.now(timezone.utc).isoformat(timespec="seconds")}, indent=2))


# ---------------- retraining flow ----------------
def retrain_on_new_data(fd, current_train_raw, new_batch_raw, test_raw, test_rul, project, client,
                        models_dir, max_rmse_increase=0.0, max_critical_rmse_increase=0.5):
    """New run-to-failure data arrived -> validate -> drift check -> retrain challenger ->
    compare with champion on the golden holdout -> promote + deploy only if not worse."""
    log = {"subset": fd, "started_at": datetime.now(timezone.utc).isoformat(timespec="seconds")}

    v = validate_batch(new_batch_raw)
    log["validation"] = v
    if not v["ok"]:
        log["decision"] = "REJECTED_BAD_DATA"
        return log

    new = new_batch_raw.copy()
    new["unit"] = new["unit"] + int(current_train_raw["unit"].max())     # keep engine ids unique
    new["RUL"] = new.groupby("unit")["cycle"].transform("max") - new["cycle"]
    combined = pd.concat([current_train_raw, new], ignore_index=True)

    champ = get_champion(client, fd)
    champ_prep = joblib.load(Path(models_dir) / f"preprocessor_{fd}.joblib")
    drift = drift_report(champ_prep, current_train_raw, new)
    log["drift_max_psi"] = float(drift.max())
    log["drift_top"] = drift.head(3).to_dict()

    res = train_and_log(fd, combined, test_raw, test_rul, project,
                        run_name=f"{fd}-retrain-{datetime.now():%Y%m%d-%H%M%S}",
                        tags={"trigger": "new_data", "stage": "challenger"},
                        extra_params={"new_engines": v["engines"], "drift_max_psi": round(float(drift.max()), 3)})
    ch = res["metrics"]
    log["challenger"] = {"version": res["version"], "RMSE": ch["RMSE"], "RMSE_RUL_le_30": ch["RMSE_RUL_le_30"]}

    if champ is None:
        better, reason = True, "no champion yet"
    else:
        c_rmse = champ["metrics"]["test_RMSE"]
        c_crit = champ["metrics"]["test_RMSE_RUL_le_30"]
        log["champion"] = {"version": champ["version"], "RMSE": c_rmse, "RMSE_RUL_le_30": c_crit}
        better = (ch["RMSE"] <= c_rmse + max_rmse_increase) and \
                 (ch["RMSE_RUL_le_30"] <= c_crit + max_critical_rmse_increase)
        reason = (f"challenger RMSE {ch['RMSE']} vs champion {c_rmse}; "
                  f"critical-zone RMSE {ch['RMSE_RUL_le_30']} vs {c_crit}")

    with mlflow.start_run(run_id=res["run_id"]):
        mlflow.set_tag("promotion_decision", "PROMOTED" if better else "KEPT_CHAMPION")
        mlflow.log_dict({k: v for k, v in log.items() if k != "validation"}, "retraining_log.json")

    if better:
        set_champion(client, fd, res["version"])
        deploy(res, fd, models_dir)
        log["decision"] = f"PROMOTED v{res['version']} to champion and deployed ({reason})"
    else:
        log["decision"] = f"KEPT champion v{champ['version']} ({reason})"
    return log

Writing ../src/mlops.py


In [3]:
from mlops import setup_mlflow, train_and_log, set_champion, deploy, retrain_on_new_data, EXPERIMENT

client = setup_mlflow(PROJECT)
rows = []
for fd in SUBSETS:
    train, test, rul = data[fd]
    train = train.drop(columns=["subset"])
    if fd == "FD001":                       # demo: FD001 starts with only 80 of its 100 engines
        train = train[train["unit"] <= 80]
    res = train_and_log(fd, train, test, rul, PROJECT, run_name=f"{fd}-initial",
                        tags={"trigger": "initial", "stage": "champion", "promotion_decision": "INITIAL_CHAMPION"})
    set_champion(client, fd, res["version"])
    deploy(res, fd, MODELS)
    rows.append({"subset": fd, "version": res["version"], "train_engines": train["unit"].nunique(),
                 **{k: res["metrics"][k] for k in ["RMSE", "RMSE_RUL_le_30", "NASA_score"]}})
    print(fd, "logged + registered + deployed", flush=True)
pd.DataFrame(rows).set_index("subset")

2026/10/30 19:09:35 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/30 19:09:35 INFO mlflow.store.db.utils: Updating database tables


FD001 logged + registered + deployed


Successfully registered model 'teliport_rul_FD001'.
Created version '1' of model 'teliport_rul_FD001'.


FD002 logged + registered + deployed


Successfully registered model 'teliport_rul_FD002'.
Created version '1' of model 'teliport_rul_FD002'.


FD003 logged + registered + deployed


Successfully registered model 'teliport_rul_FD003'.
Created version '1' of model 'teliport_rul_FD003'.


FD004 logged + registered + deployed


Successfully registered model 'teliport_rul_FD004'.
Created version '1' of model 'teliport_rul_FD004'.


,version,train_engines,RMSE,RMSE_RUL_le_30,NASA_score
subset,,,,,
FD001,1,80,12.51,3.61,225.7
FD002,1,260,23.20,2.78,4744.5
FD003,1,100,12.25,2.35,261.0
FD004,1,249,23.83,4.41,3365.4


In [4]:
INCOMING.mkdir(parents=True, exist_ok=True)
train, test, rul = data["FD001"]
batch = train[train["unit"] > 80].drop(columns=["RUL", "subset"]).copy()
batch["unit"] = batch["unit"] - 80                  # new fleet numbers its engines from 1
batch.to_csv(INCOMING / "FD001_batch_001.csv", index=False)
print("New batch file:", INCOMING / "FD001_batch_001.csv",
      "| engines:", batch["unit"].nunique(), "| rows:", len(batch))

New batch file: C:\Users\hp\Documents\teliport_prototype\data\incoming\FD001_batch_001.csv | engines: 20 | rows: 4493


In [5]:
new_batch = pd.read_csv(INCOMING / "FD001_batch_001.csv")
current = train[train["unit"] <= 80].drop(columns=["subset"])
t0 = time.time()
log = retrain_on_new_data("FD001", current, new_batch, test, rul, PROJECT, client, MODELS)
print(f"Retraining flow finished in {time.time() - t0:.1f}s\n")
print(json.dumps(log, indent=2, default=str))

Retraining flow finished in 8.7s

{
  "subset": "FD001",
  "started_at": "2026-10-30T13:41:14+00:00",
  "validation": {
    "ok": true,
    "issues": [],
    "rows": 4493,
    "engines": 20
  },
  "drift_max_psi": 0.12,
  "drift_top": {
    "s14": 0.12,
    "s9": 0.077,
    "s13": 0.019
  },
  "challenger": {
    "version": "2",
    "RMSE": 11.34,
    "RMSE_RUL_le_30": 2.73
  },
  "champion": {
    "version": 1,
    "RMSE": 12.51,
    "RMSE_RUL_le_30": 3.61
  },
  "decision": "PROMOTED v2 to champion and deployed (challenger RMSE 11.34 vs champion 12.51; critical-zone RMSE 2.73 vs 3.61)"
}


Registered model 'teliport_rul_FD001' already exists. Creating a new version of this model...
Created version '2' of model 'teliport_rul_FD001'.


In [6]:
bad = new_batch.drop(columns=["s11"])
bad_log = retrain_on_new_data("FD001", current, bad, test, rul, PROJECT, client, MODELS)
print("Decision:", bad_log["decision"])
print("Issues  :", bad_log["validation"]["issues"])

Decision: REJECTED_BAD_DATA
Issues  : ["missing columns: ['s11']"]


In [7]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENT], order_by=["attributes.start_time ASC"])
cols = {"tags.mlflow.runName": "run", "params.subset": "subset", "params.train_engines": "train_engines",
        "metrics.test_RMSE": "RMSE", "metrics.test_RMSE_RUL_le_30": "RMSE_RUL<=30",
        "metrics.test_NASA_score": "NASA_score", "tags.promotion_decision": "decision"}
history = runs[list(cols)].rename(columns=cols)
history.to_csv(REPORTS / "mlflow_run_history.csv", index=False)
display(history)

for fd in SUBSETS:
    mv = client.get_model_version_by_alias(f"teliport_rul_{fd}", "champion")
    print(f"{fd}: champion = v{mv.version}  (run {mv.run_id[:8]})")

,run,subset,train_engines,RMSE,RMSE_RUL<=30,NASA_score,decision
0,FD001-initial,FD001,80,12.51,3.61,225.7,INITIAL_CHAMPION
1,FD002-initial,FD002,260,23.20,2.78,4744.5,INITIAL_CHAMPION
2,FD003-initial,FD003,100,12.25,2.35,261.0,INITIAL_CHAMPION
3,FD004-initial,FD004,249,23.83,4.41,3365.4,INITIAL_CHAMPION
4,FD001-retrain-20261030-191114,FD001,100,11.34,2.73,177.1,PROMOTED


FD001: champion = v2  (run a9c189d0)
FD002: champion = v1  (run 22090160)
FD003: champion = v1  (run 0156c166)
FD004: champion = v1  (run 076f5108)
